# BoVW + CNN Two-Stage Classifier Training

Trains the two-stage answer cell classifier described in:
> Afifi & Hussain, *The achievement of higher flexibility in multiple-choice-based tests
> using image classification techniques*, IJDAR 22, 127–142 (2019). Section 4.2, Strategy B.

**Training data:** `cells_exam5.npz` — exam5 subset (solid-filled marking style only).
Filtering to a single marking style gives a much cleaner confirmed/crossedout separation
than training on the full mixed dataset.

**Architecture:**
- **Stage 1** — BoVW + SVM: SIFT descriptors → K-means vocabulary (k=200) → LinearSVC → *filled* or *empty*
- **Stage 2** — CNN: 3-conv network (64×64 grayscale) → *confirmed* or *crossed-out*

---

## Before running

Set up one **Colab Secret** (🔑 icon in the left sidebar, enable *Notebook access*):

| Secret name | Value |
|---|---|
| `HF_TOKEN` | Your HuggingFace **write** token — [hf.co/settings/tokens](https://huggingface.co/settings/tokens) |

The training dataset (`cells_exam5.npz`) is loaded directly from HuggingFace — no Drive upload needed.

In [ ]:
# ── 1  Install dependencies ────────────────────────────────────────────────────
!pip install -q "scikit-learn==1.7.2" torch torchvision huggingface_hub opencv-python-headless tqdm onnxscript onnx

In [ ]:
# ── 2  Read Colab secrets ──────────────────────────────────────────────────────
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
HF_REPO  = "spraxx/exam-answer-classifier"

print(f"HF_REPO : {HF_REPO}")
print(f"HF_TOKEN: {'*' * 6}{HF_TOKEN[-4:] if HF_TOKEN else 'NOT SET'}")

In [ ]:
# ── 3  Load cells_exam5.npz from HuggingFace Hub ─────────────────────────────
from huggingface_hub import hf_hub_download
import numpy as np

npz_path = hf_hub_download(
    repo_id="spraxx/exam-answer-cells",
    filename="cells_exam5.npz",
    repo_type="dataset",
    token=HF_TOKEN,
)

data   = np.load(npz_path, allow_pickle=True)
images = data["images"]   # (N, 64, 64) uint8
labels = data["labels"]   # (N,)  1=confirmed  2=crossedout  3=empty

print(f"Loaded {len(images):,} cells from spraxx/exam-answer-cells (exam5 subset)")
for lbl, name in [(1, 'confirmed'), (2, 'crossedout'), (3, 'empty')]:
    print(f"  {name:12s}: {(labels == lbl).sum():>6,}")

---
## Stage 1 — Bag of Visual Words + SVM

SIFT (Lowe, IJCV 2004) → K-means visual vocabulary → L1-normalised histogram → RBF SVM (Vapnik, 1998).

Binary target: `confirmed + crossedout → filled (1)` | `empty → empty (0)`.

In [ ]:
# ── 4  Extract SIFT descriptors ────────────────────────────────────────────────
import cv2
from tqdm.auto import tqdm

sift = cv2.SIFT_create()
all_descriptors  = []   # pooled — used to fit K-means
per_image_descs  = []   # per-image — used to build histograms

for img in tqdm(images, desc="SIFT"):
    _, desc = sift.detectAndCompute(img, None)
    per_image_descs.append(desc)
    if desc is not None and len(desc) > 0:
        all_descriptors.append(desc)

all_descriptors = np.vstack(all_descriptors).astype(np.float32)
print(f"Pooled descriptors: {len(all_descriptors):,}  shape={all_descriptors.shape}")

In [ ]:
# ── 5  Fit K-means visual vocabulary (k=200) ───────────────────────────────────
from sklearn.cluster import MiniBatchKMeans

K = 200
kmeans = MiniBatchKMeans(n_clusters=K, random_state=42, batch_size=4096, n_init=3, verbose=1)
kmeans.fit(all_descriptors)
print(f"\nVocabulary ready: {K} visual words")

In [ ]:
# ── 6  Build per-image BoVW histograms ─────────────────────────────────────────
def build_histogram(desc, kmeans, k):
    hist = np.zeros(k, dtype=np.float32)
    if desc is not None and len(desc) > 0:
        word_ids = kmeans.predict(desc)
        np.add.at(hist, word_ids, 1)
        total = hist.sum()
        if total > 0:
            hist /= total          # L1 normalise
    return hist

histograms = np.array(
    [build_histogram(d, kmeans, K) for d in tqdm(per_image_descs, desc="Histograms")],
    dtype=np.float32,
)
print(f"Histograms: {histograms.shape}")

In [ ]:
# ── 7  Train Stage 1 SVM with 5-fold CV ───────────────────────────────────────
# LinearSVC instead of RBF SVC: O(n) vs O(n²) — trains in seconds on 32K samples.
# BoVW histograms are compact float vectors that are typically linearly separable
# after L1 normalisation, so accuracy is comparable to RBF in practice.
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score

binary_labels = (labels != 3).astype(int)   # 1=filled, 0=empty

base_svm = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(base_svm, histograms, binary_labels, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Stage 1 — 5-fold CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"Per-fold: {[f'{s:.4f}' for s in scores]}")

# Wrap with Platt scaling so .predict_proba() is available if needed later
svm = CalibratedClassifierCV(base_svm, cv=5)
svm.fit(histograms, binary_labels)
print("\nSVM fitted on full dataset")

---
## Stage 2 — CNN (confirmed vs crossed-out)

3-conv CNN for 64×64 grayscale input, 2-class output.

In the exam5 subset: 89 crossedout samples vs 6,921 confirmed.
Following the paper's augmentation protocol (Afifi & Hussain §5): translate ±{2,4}px,
rotate ±{1,2,3}°, horizontal/vertical flip.

In [ ]:
# ── 8  CNN architecture ────────────────────────────────────────────────────────
import torch
import torch.nn as nn

class AnswerCellCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # → 32×32
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # → 16×16
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # →  8× 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Params: {sum(p.numel() for p in AnswerCellCNN().parameters()):,}")

In [ ]:
# ── 9  Build augmented Stage 2 dataset ────────────────────────────────────────
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split

def augment_crossedout(img_np):
    """Yield augmented variants of one 64×64 uint8 grayscale image."""
    t = TF.to_tensor(img_np)          # (1, 64, 64) float32 [0, 1]
    yield t
    for dx in [-4, -2, 2, 4]:
        for dy in [-4, -2, 2, 4]:
            yield TF.affine(t, angle=0, translate=[dx, dy], scale=1.0, shear=0)
    for angle in [-3, -2, -1, 1, 2, 3]:
        yield TF.rotate(t, angle)
    yield TF.hflip(t)
    yield TF.vflip(t)

class Stage2Dataset(Dataset):
    """confirmed → class 0 | crossedout → class 1 (paper's CNN label scheme)."""
    def __init__(self, imgs, lbls):
        self.samples = []
        for img, lbl in zip(imgs, lbls):
            cls = 0 if lbl == 1 else 1
            if lbl == 2:             # crossedout — augment
                for aug in augment_crossedout(img):
                    self.samples.append((aug, cls))
            else:
                self.samples.append((TF.to_tensor(img), cls))

    def __len__(self):        return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

mask      = labels != 3
s2_imgs   = images[mask]
s2_labels = labels[mask]

full_ds = Stage2Dataset(s2_imgs, s2_labels)
counts  = {c: sum(1 for _, y in full_ds if y == c) for c in [0, 1]}
print(f"Stage 2 dataset  confirmed={counts[0]:,}  crossedout={counts[1]:,}  total={len(full_ds):,}")

idx        = list(range(len(full_ds)))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)
train_loader = DataLoader(Subset(full_ds, train_idx), batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(Subset(full_ds, val_idx),   batch_size=64, shuffle=False, num_workers=2)
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

In [ ]:
# ── 10  Train Stage 2 CNN (30 epochs) ─────────────────────────────────────────
model     = AnswerCellCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(30):
    # --- train ---
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.clone().detach().to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # --- validate ---
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.clone().detach().to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/30  loss={train_loss/len(train_loader):.4f}  val_acc={correct/total:.4f}")

In [ ]:
# ── 11  Export CNN → ONNX (single self-contained file) ────────────────────────
# The dynamo exporter splits weights into an external .data file by default.
# We consolidate back into one file so a single upload to HuggingFace suffices.
import onnx

model.eval().cpu()
dummy = torch.zeros(1, 1, 64, 64)

torch.onnx.export(
    model, dummy, "cnn_classifier.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
)

# Merge external data (if any) into a single self-contained file
proto = onnx.load("cnn_classifier.onnx")
onnx.save(proto, "cnn_classifier.onnx", save_as_external_data=False)
print("Saved cnn_classifier.onnx (self-contained)")

In [ ]:
# ── 12  Save BoVW model ────────────────────────────────────────────────────────
import joblib

joblib.dump({"kmeans": kmeans, "svm": svm}, "bovw_svm.pkl")
print("Saved bovw_svm.pkl")

In [ ]:
# ── 13  Upload both models to HuggingFace Hub ──────────────────────────────────
# Credentials come from Colab Secrets — no token visible in the notebook.
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)

for fname in ["bovw_svm.pkl", "cnn_classifier.onnx"]:
    url = api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=HF_REPO,
    )
    print(f"  ✓ {fname}  →  {url}")

print(f"""
Done. To activate the classifier locally, edit config/default.yaml:

  bubble:
    ml_classifier:
      enabled: true
      hf_repo: "{HF_REPO}"
""")